In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
import re

In [2]:
def create_atomic_electron_config_database():
    df_pde = pd.read_csv('./datasets/PubChemElements_all.csv', usecols=['Symbol', 'ElectronConfiguration'])
    num_elements = len(df_pde['Symbol'])
    max_shell_config = {'1s': 0, 
                        '2s': 0, 
                        '2p': 0, 
                        '3s': 0, 
                        '3p': 0, 
                        '4s': 0, 
                        '3d': 0, 
                        '4p': 0, 
                        '5s': 0, 
                        '4d': 0, 
                        '5p': 0, 
                        '6s': 0, 
                        '4f': 0, 
                        '5d': 0, 
                        '6p': 0, 
                        '7s': 0, 
                        '5f': 0, 
                        '6d': 0, 
                        '7p': 0}
    outer_shell_config = {'He':0, 'Ne':0, 'Ar':0, 'Kr':0, 'Xe':0, 'Rn':0, 's':0, 'p':0, 'd':0, 'f':0}
    num_max_shells = len(max_shell_config)
    electronic_database = np.zeros((num_elements,num_max_shells))
    compact_electronic_config_db = np.zeros((num_elements, len(outer_shell_config), len(outer_shell_config)))
    database_dict = {}
    for i in range(num_elements):
        atomic_electron_config = {}
        outer_shell = deepcopy(outer_shell_config)
        core_match = re.search(r'\[([A-Za-z]+)\]', df_pde['ElectronConfiguration'].iloc[i])
        if core_match:
            core_element = core_match.group(1)
            atomic_electron_config |= database_dict[core_element]
            outer_shell[core_element] = 1
        matches = re.findall(r'(\d[spdfg])(\d+)', df_pde['ElectronConfiguration'].iloc[i])
        for shell, electrons in matches:
            atomic_electron_config[shell] = int(electrons)
            outer_shell[shell[1]] = int(electrons)
        database_dict[df_pde['Symbol'].iloc[i]] = atomic_electron_config
        template = deepcopy(max_shell_config)
        for shell in atomic_electron_config:
            template[shell] = atomic_electron_config[shell]
        for shellnum in range(num_max_shells):
            electronic_database[i, shellnum] = list(template.values())[shellnum]
        compact_config = np.array(list(outer_shell.values()))
        for compact_db_index in range(len(outer_shell_config)):
            compact_electronic_config_db[i, 0, compact_db_index] = compact_config[compact_db_index]
        for k in range(1, len(outer_shell_config)):
            compact_electronic_config_db[i, k] = np.roll(compact_electronic_config_db[i, k-1], 1)
    return compact_electronic_config_db, electronic_database, database_dict

In [3]:
# compact_np_db, full_np_db, database_dict = create_atomic_electron_config_database()
compact_np_db, _, _ = create_atomic_electron_config_database()
# for key in database_dict:
#     print(f"{key}={database_dict[key]}")

In [4]:
with np.printoptions(threshold=np.inf, linewidth=400):
    print(compact_np_db.shape)
    # print(full_np_db)
    # print(database_dict)

(118, 10, 10)
